# 07 (Kaggle) — 2x2 evaluation: M0 vs M1, repair off vs on

|        | repair off | repair on |
|--------|-----------|-----------|
| **M0** | A | B |
| **M1** | C | D |

`curriculum_effect = C - A`, `repair_effect = B - A`, interaction = the rest.

**Sized for a single Kaggle session, not the full paper-scale spec.** The
original design (`n=20` samples, `k_repair=3`, 3 seeds, full combined eval
sets -- a few hundred problems) would take many hours to days at this
project's measured generation throughput (~28s/batch of 8, and the
repair-on cell is *unbatched* -- one problem at a time, up to `k_repair+1`
sequential generate() calls each). Defaults below: **30 problems, n=5,
1 seed** -- roughly 1.5-2 hours total. Raise `LIMIT`/`N`/`SEEDS` below if
more GPU-hour budget is available later; nothing about the code requires
these particular numbers.

**Before running:**
1. Attach `verilog-slm-data`, `verilog-slm-m0-retrain-final` (M0's actual
   trained weights -- must exist as a real dataset with `adapter_model.
   safetensors` in it, not just the metadata-only `verilog-slm-m0-retrain-
   diag`), and `verilog-slm-m1-retrain-final`.
2. Accelerator = GPU, Internet = On.
3. `verify()` needs the real testbench-simulate stage this time (unlike
   the diagnostic pass's `verify_static()`), so the toolchain cell below
   (`iverilog` required, `yosys`/`verible` optional) actually matters
   here -- don't skip it.

In [ ]:
# --- Bootstrap: repo + eval sets + both trained adapters ---
import os, shutil, glob

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/verilog-slm'):
    os.system(f'git clone {REPO} /kaggle/working/verilog-slm')
os.chdir('/kaggle/working/verilog-slm')
os.system('git pull')
os.makedirs('artifacts', exist_ok=True)
os.makedirs('data/eval', exist_ok=True)

for fname, dest in [
    ('verilogeval_v2.jsonl', 'data/eval/verilogeval_v2.jsonl'),
    ('rtllm_v2.jsonl', 'data/eval/rtllm_v2.jsonl'),
]:
    hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
    if hits:
        shutil.copy(hits[0], dest); print(f'restored {dest} <- {hits[0]}')
    else:
        print(f'WARNING: {fname} not found -- attach verilog-slm-data')

def local_adapter(keyword, dest):
    cfgs = [p for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True) if keyword in p]
    if not cfgs:
        print(f'WARNING: no adapter found with "{keyword}" in its path -- is that dataset attached?')
        return False
    src = os.path.dirname(sorted(cfgs, key=len)[0])
    weight_file = os.path.join(src, 'adapter_model.safetensors')
    if not os.path.exists(weight_file):
        print(f'WARNING: {src} has adapter_config.json but no adapter_model.safetensors -- '
              f'this looks like a metadata-only dataset, not the real trained weights')
        return False
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    print(f'{dest} <- {src}')
    return True

have_m0 = local_adapter('m0-retrain-final', 'artifacts/m0_final')
have_m1 = local_adapter('m1-retrain-final', 'artifacts/m1_final')
print('done: have_m0 =', have_m0, '| have_m1 =', have_m1)
if not (have_m0 and have_m1):
    print('STOP: do not proceed until both adapters are confirmed present with real weight files.')

In [ ]:
!pip install -q -r requirements.txt -r requirements-train.txt
!pip install -q -U "torchao>=0.16.0"
print('installs done')

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi -L

## RTL toolchain

`iverilog` is required for the `simulate` stage (real testbenches this
time, not the taxonomy-only `verify_static()` the diagnostic pass used).
`yosys`/`verible` are optional/soft-gated.

In [ ]:
import os
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

## Run the 2x2

`--m0-gpu-hours`/`--m1-gpu-hours` come straight from each run's own
`run_meta.json` (for the per-GPU-hour efficiency metric) -- read here
rather than hardcoded, so this stays correct if M0 gets retrained again
with a different wall-clock cost.

In [ ]:
import json, glob

# run_meta.json lives at the *run* level (artifacts/m0/run_meta.json in
# the original layout), not inside final/ -- if it was uploaded alongside
# the adapter in the same dataset (as it was for M1's), pull it in for
# the per-GPU-hour metric; else skip that metric rather than guessing.
def find_run_meta(keyword):
    hits = [p for p in glob.glob('/kaggle/input/**/run_meta.json', recursive=True) if keyword in p]
    return json.load(open(hits[0])) if hits else None

m0_meta = find_run_meta('m0-retrain')
m1_meta = find_run_meta('m1-retrain')
m0_gpu_hours = m0_meta['train_gpu_hours'] if m0_meta else None
m1_gpu_hours = m1_meta['train_gpu_hours'] if m1_meta else None
print('M0 GPU-hours:', m0_gpu_hours, '| M1 GPU-hours:', m1_gpu_hours)

In [ ]:
# Sizing -- raise these if more GPU-hour budget is available.
LIMIT = 30          # random subsample of eval problems (fixed sampling seed, same subset every run)
N = 5               # samples per problem, repair-off cell
K_REPAIR = 3
SEEDS = "1337"      # space-separated for more than one, e.g. "1337 2025"
MAX_NEW_TOKENS = 256

gpu_hours_args = ""
if m0_gpu_hours and m1_gpu_hours:
    gpu_hours_args = f"--m0-gpu-hours {m0_gpu_hours} --m1-gpu-hours {m1_gpu_hours}"

!python -m src.eval.run_eval \
  --m0-adapter artifacts/m0_final --m1-adapter artifacts/m1_final \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --n {N} --k-repair {K_REPAIR} --seeds {SEEDS} --limit {LIMIT} --max-new-tokens {MAX_NEW_TOKENS} \
  {gpu_hours_args} \
  --out artifacts/eval_report.json

## Headline table: pass@1 / pass@5, all 4 cells

With a single seed there's no cross-seed std to report -- add more
entries to `SEEDS` above and rerun for that (each extra seed roughly
doubles/triples the wall-clock cost).

In [ ]:
import numpy as np, pandas as pd
report = json.load(open('artifacts/eval_report.json'))
cell_names = ['M0_repair_off', 'M0_repair_on', 'M1_repair_off', 'M1_repair_on']
rows = []
for cell in cell_names:
    p1 = np.array([s['cells'][cell]['pass@1']['mean'] for s in report['per_seed']])
    p5 = np.array([s['cells'][cell]['pass@5']['mean'] for s in report['per_seed']])
    rows.append({'cell': cell, 'pass@1_mean': p1.mean(), 'pass@1_std': p1.std(),
                 'pass@5_mean': p5.mean(), 'pass@5_std': p5.std()})
pd.DataFrame(rows)

## Effects: curriculum, repair, interaction

In [ ]:
effects = [s['effects'] for s in report['per_seed']]
pd.DataFrame(effects)

## Per-category delta table
M0 vs M1, repair-off cells only (cleanest comparison, no repair-loop
confound). Uses seed[0]'s taxonomy tables.

In [ ]:
from src.eval.metrics import per_category_delta_table
TARGETED = {'incomplete_sensitivity', 'missing_default_case'}
def expand(table):
    rows = []
    for row in table:
        rows += [{'ok': False, 'error_label': row['label']}] * row['count']
    return rows
m0_rows = expand(report['per_seed'][0]['cells']['M0_repair_off']['taxonomy_table'])
m1_rows = expand(report['per_seed'][0]['cells']['M1_repair_off']['taxonomy_table'])
delta_table = per_category_delta_table(m0_rows, m1_rows, TARGETED)
pd.DataFrame(delta_table).sort_values('delta')

## Repair-loop diagnostics

In [ ]:
for cell in ['M0_repair_on', 'M1_repair_on']:
    diag = report['per_seed'][0]['cells'][cell]['repair_diagnostics']
    print(cell, '->', json.dumps(diag, indent=2))

## Efficiency: pass@1 per GPU-hour

In [ ]:
print(json.dumps(report.get('efficiency', {}), indent=2))

## Catch-all share sanity check
If any cell exceeds 60%, the taxonomy isn't discriminating well for that
cell's results -- read the pass@k numbers with that caveat.

In [ ]:
for cell in cell_names:
    share = report['per_seed'][0]['cells'][cell]['catch_all_share']
    flag = ' <-- taxonomy not discriminating well' if share > 0.6 else ''
    print(f"{cell}: catch_all_share={share:.1%}{flag}")

## End of session

Quick Save, then push `artifacts/eval_report.json` (small) as a new
dataset or add it to an existing one -- this is your actual result, don't
lose it the way M0's first retrain was lost.